<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# 🎧 AfriCare Support Analytics — DataProjectLab

## 📓 Notebook 2 — Data Cleaning & Feature Engineering

---

> **🎯 Niveau :** Avancé | **⏱️ Durée :** 3-4h
> ❗ Ce notebook ne contient pas de solution.

### Informations du notebook
- **Objectif** : produire un dataset propre et enrichi pour l'analyse SQL et le modèle ML
- **Sortie attendue** : `support_clean_analytics.csv` + table SQL `fact_support_analytics`

In [ ]:
# ================================
# 0. Imports & configuration
# ================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

import os
import sys

# Détecter si on est dans Colab ou en local
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/DataProjectLab/projects/customer_support_analytics/"
else:
    SAVE_PATH = "./outputs/"

# Chemin pour enregistrer les fichiers exportés
os.makedirs(SAVE_PATH, exist_ok=True)
print(f" Environnement Colab : {'Colab' if IN_COLAB else 'Local'}")
print(f" Dossier de travail : {SAVE_PATH}")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

print("✅ Environnement prêt")

---

# 1. Chargement des données

Charge les 5 fichiers. Pense à parser les colonnes de dates.

In [ ]:
# Chargement des données


BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/main/projets/customer_support_analytics/data/"


#agents     = pd.read_csv(BASE_URL + "agents.csv")
#categories = pd.read_csv(BASE_URL + "categories.csv")
#tickets    = pd.read_csv(BASE_URL + "tickets.csv")
#interactions = pd.read_csv(BASE_URL + "interactions.csv")
#sla_alerts = pd.read_csv(BASE_URL + "sla_alerts.csv")

print("agents      :", agents.shape)
print("categories  :", categories.shape)
print("tickets     :", tickets.shape)
print("interactions:", interactions.shape)
print("sla_alerts  :", sla_alerts.shape)

---

# 2. Audit qualité rapide

Produis un tableau récapitulatif (lignes, colonnes, valeurs manquantes, doublons) pour chacun des 5 datasets.

> 💡 Utilise une boucle sur un dict `{"nom": dataframe}` pour automatiser l'affichage.

In [ ]:
# Rapport qualité automatisé
# Votre code ici

In [ ]:
# Types de données — vérification des dates et numériques
# Votre code ici

In [ ]:
# Valeurs manquantes par colonne sur tickets (détail)
# Votre code ici

In [ ]:
# Cohérence des clés et contrôles métier
# Agents inconnus dans tickets
# Catégories inconnues dans tickets
# Montants / délais aberrants

# Votre code ici

---

# 3. Nettoyage

## Logique à appliquer

- Les `ticket_id` doivent être uniques dans tickets
- Les délais négatifs (`resolution_heures < 0`) sont incohérents → remplacer par NaN puis médiane
- Les CSAT hors [1-5] sont exclus
- Le statut est normalisé (strip des espaces)
- Les doublons sont supprimés dans toutes les tables

🔹 3.1 Doublons

In [ ]:
# Suppression des doublons dans toutes les tables
# Votre code ici

🔹 3.2 Anomalies métier

In [ ]:
# Délais négatifs → NaN → médiane
# Votre code ici

In [ ]:
# Normalisation du statut (strip)
# Votre code ici

---

# 4. Construction de la table analytique

## 4.1 Fusion principale

Fusionne `tickets` avec `agents` (colonnes utiles) et `categories` (colonnes utiles).

> 💡 `df = tickets.merge(agents[...], on='agent_id', how='left').merge(categories[...], on='category_id', how='left')`

In [ ]:
# Fusion principale tickets + agents + categories
# Votre code ici

df = 

## 4.2 Variables temporelles

In [ ]:
# Extraire : heure_creation, jour_semaine, est_weekend, heure_hors_bureau (avant 8h ou après 18h)
# Votre code ici

## 4.3 Variable tier numérique

In [ ]:
# tier_num : Tier 1 → 1, Tier 2 → 2, Tier 3 → 3
# Votre code ici

## 4.4 Variable cible ML ⭐

Un ticket est **à risque** (`ticket_at_risk = 1`) si au moins une des conditions suivantes est vraie :
- `sla_breach == 1`
- `in_backlog == 1`
- `statut == 'Escaladé'`

> ⚠️ Règle d'or : n'utilise jamais `resolution_heures`, `sla_breach`, `csat` comme feature ML —
> ces valeurs ne sont connues qu'APRÈS la résolution du ticket. Ce serait du data leakage !

In [ ]:
# Création de la variable cible ticket_at_risk
# Votre code ici

df["ticket_at_risk"] = 

print("Taux de tickets à risque :", round(df["ticket_at_risk"].mean() * 100, 1), "%")

## 4.5 Flags business utiles

In [ ]:
# is_high_priority : priorite <= 2
# is_long_wait      : first_response_heures > 4
# is_reopened       : reopened == 1

# Votre code ici

---

# 5. Sélection des colonnes finales

Sélectionne les colonnes pertinentes pour l'analyse et le ML.

Colonnes recommandées :
`ticket_id, created_at, category_id, agent_id, pays, canal, priorite, statut, sla_heures,
first_response_heures, resolution_heures, sla_breach, nb_contacts, reopened, csat, in_backlog,
ratio_sla, nom_agent, tier, tier_num, bureau, csat_moyen, taux_resolution_pct, tickets_actifs,
nom_cat, domaine, priorite_defaut, sla_strict, heure_creation, jour_semaine, est_weekend,
heure_hors_bureau, ticket_at_risk, is_high_priority, is_long_wait, is_reopened`

In [ ]:
# Sélection et vérification des colonnes finales
# Votre code ici

final_df = 

print("Shape finale :", final_df.shape)
final_df.head()

---

# 6. Export CSV

Exporte le dataset propre pour les notebooks suivants.

In [ ]:
# Export CSV
final_df.to_csv(f'{SAVE_PATH}support_clean_analytics.csv', index=False)
print("✅ Fichier exporté : support_clean_analytics.csv")

---

# 📌 Conclusion

👉 Résumer :

- Quels problèmes ont été corrigés ?
- Quelles variables ont été créées et pourquoi ?
- Quel est le taux de tickets à risque ? Ce ratio est-il équilibré ?

---

💡 Le dataset final est maintenant prêt pour :
- l'analyse SQL (Notebook 3)
- le dashboard Power BI (Notebook 4)
- le modèle ML (Notebook 5)

➡️ **Prochaine étape : Notebook 3 — SQL Analytics, KPIs & Performance**

---

*DataProjectLab — apprendre la data sur des cas concrets, structurés et orientés métier.*